# Supplementary Figure 2 — single-cell localization of donor-bulk CLAMP LVs


In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(cowplot)
  library(ggrastr)
  library(ggrepel)
  library(grid)
  library(svglite)
  library(ragg)
})

dir.create(dirname(snakemake@output[["pdf"]]), recursive=TRUE, showWarnings=FALSE)

FIG_W <- 183
FIG_H <- 285
FONT_FAMILY <- "Helvetica"
FS_TAG <- 9
FS_TITLE <- 6.5
FS_AXIS_TITLE <- 6.5
FS_AXIS_TEXT <- 5
LINE_W <- 0.25

DATASET_LEVELS <- c("Heart_Datar2026", "PBMC_1k1k",
  "Brain_Mathys2023", "Brain_Xiong2023",
  "Lung_Sikkema2023", "PBMC_Perez2022")
DATASET_LABELS <- c(Heart_Datar2026="Heart: Datar", PBMC_1k1k="PBMC: 1k1k",
  Brain_Mathys2023="Brain: Mathys", Brain_Xiong2023="Brain: Xiong",
  Lung_Sikkema2023="Lung: Sikkema", PBMC_Perez2022="PBMC: Perez")
DATASET_COLORS <- c(Brain_Mathys2023="#5B8FF9", Brain_Xiong2023="#9270CA",
  Heart_Datar2026="#E8684A", PBMC_1k1k="#5AD8A6",
  PBMC_Perez2022="#F6BD16", Lung_Sikkema2023="#6DC8EC")
KNOWN_CELL_COLORS <- c(B_cell="#4477AA", Myeloid="#EE6677",
  NK="#228833", T_cell="#CCBB44", CD4_T="#66CCEE",
  CD8_T="#AA3377", CD14_Mono="#EE7733", CD16_Mono="#0077BB",
  DC="#BBBBBB", Plasma_B="#EE3377", gd_T="#009988")

purity <- fread(snakemake@input[["purity"]])
cells <- fread(snakemake@input[["umap_cells"]])
lvs <- fread(snakemake@input[["umap_lvs"]])
activity_columns <- paste0("activity_", 1:4)
stopifnot(setequal(unique(cells$dataset), DATASET_LEVELS),
          setequal(unique(lvs$dataset), DATASET_LEVELS),
          all(activity_columns %in% names(cells)),
          lvs[, all(.N == 4L), by=dataset]$V1,
          cells[, all(uniqueN(cell_index) == .N), by=dataset]$V1,
          all(is.finite(as.matrix(cells[, c("umap1", "umap2", activity_columns), with=FALSE]))))

all_cell_types <- sort(unique(cells$mapped_cell_type))
CELL_COLORS <- setNames(grDevices::hcl.colors(length(all_cell_types), "Dark 3"),
                        all_cell_types)
known <- intersect(names(KNOWN_CELL_COLORS), names(CELL_COLORS))
CELL_COLORS[known] <- KNOWN_CELL_COLORS[known]
pretty_cell_type <- function(x) gsub("_", " ", x, fixed=TRUE)

theme_nm <- function() {
  theme_classic(base_size=FS_AXIS_TEXT, base_family=FONT_FAMILY) %+replace%
    theme(axis.line=element_line(linewidth=LINE_W, colour="black"),
      axis.ticks=element_line(linewidth=LINE_W, colour="black"),
      axis.text=element_text(size=FS_AXIS_TEXT, colour="black"),
      axis.title=element_text(size=FS_AXIS_TITLE, colour="black"),
      panel.grid=element_blank(), plot.margin=margin(1,1,1,1,"mm"))
}
add_tag <- function(p, label) {
  ggdraw() + draw_plot(p) +
    draw_label(label, x=0, y=1, hjust=0, vjust=1, size=FS_TAG,
               fontface="bold", fontfamily=FONT_FAMILY)
}

# A: top-1% annotated-cell purity across all matched LVs.
purity[, dataset := factor(dataset, levels=DATASET_LEVELS)]
purity[, dataset_label := factor(DATASET_LABELS[as.character(dataset)],
                                 levels=DATASET_LABELS[DATASET_LEVELS])]
panel_A <- ggplot(purity, aes(dataset_label, recovery_pct, fill=dataset)) +
  geom_boxplot(width=0.58, outlier.shape=NA, linewidth=0.28, alpha=0.92) +
  geom_jitter(width=0.10, size=0.70, shape=21, fill="#333333",
              colour="#333333", stroke=0, alpha=0.75) +
  scale_fill_manual(values=DATASET_COLORS, drop=FALSE) +
  scale_y_continuous(limits=c(0, 100), breaks=seq(0,100,25),
                     expand=expansion(mult=c(0.01,0.04))) +
  labs(x=NULL, y="Top-1% annotated-cell purity (%)") +
  theme_nm() +
  theme(legend.position="none",
        axis.text.x=element_text(angle=25, hjust=1, size=5.5))

map_theme <- theme_void(base_family=FONT_FAMILY, base_size=4.2) +
  theme(panel.border=element_rect(colour="black", fill=NA, linewidth=0.25),
        plot.title=element_text(size=4.0, hjust=0.5, lineheight=0.92,
                                margin=margin(0,0,0.35,0,"mm")),
        plot.margin=margin(0.35,0.35,0.35,0.35,"mm"))

feature_sheet <- function(dataset_id) {
  d <- cells[dataset == dataset_id]
  meta <- lvs[dataset == dataset_id][order(plot_order)]
  stopifnot(nrow(meta) == 4L, nrow(d) > 0L)
  centroids <- d[, .(umap1=median(umap1), umap2=median(umap2)),
                 by=mapped_cell_type]
  centroids[, label := pretty_cell_type(mapped_cell_type)]
  annotation <- ggplot(d, aes(umap1, umap2, colour=mapped_cell_type)) +
    ggrastr::rasterise(geom_point(size=0.16, alpha=0.78), dpi=300) +
    ggrepel::geom_text_repel(data=centroids, aes(label=label),
      colour="black", size=1.20, fontface="bold", family=FONT_FAMILY,
      seed=123, box.padding=0.10, point.padding=0.05, min.segment.length=0,
      segment.size=0.16, max.overlaps=Inf, show.legend=FALSE) +
    scale_colour_manual(values=CELL_COLORS) + coord_equal() +
    labs(title="Cell-type annotation") + map_theme +
    theme(legend.position="none",
          plot.title=element_text(size=4.5, face="bold"))
  features <- lapply(seq_len(4L), function(i) {
    m <- meta[plot_order == i]
    high <- CELL_COLORS[[m$cell_type]]
    score_col <- paste0("activity_", i)
    ggplot(d, aes(umap1, umap2, colour=.data[[score_col]])) +
      ggrastr::rasterise(geom_point(size=0.16), dpi=300) +
      scale_colour_gradientn(colours=c("#D8E2EF", "white", high)) +
      coord_equal() +
      labs(title=sprintf("%s · %s\nRecovery = %.1f%%",
        pretty_cell_type(m$cell_type), m$LV, m$recovery_pct)) +
      map_theme + theme(legend.position="none")
  })
  feature_grid <- plot_grid(plotlist=features, ncol=2, align="hv")
  map_row <- plot_grid(annotation, feature_grid, nrow=1,
                       rel_widths=c(0.44, 0.56))
  header <- ggdraw() + draw_label(DATASET_LABELS[[dataset_id]],
    x=0.5, y=0.45, hjust=0.5, size=FS_TITLE, fontface="bold",
    fontfamily=FONT_FAMILY)
  plot_grid(header, map_row, ncol=1, rel_heights=c(0.08, 0.92))
}

cohort_sheets <- lapply(DATASET_LEVELS, feature_sheet)
tagged_sheets <- Map(add_tag, cohort_sheets, LETTERS[2:7])
umap_grid <- plot_grid(plotlist=tagged_sheets, ncol=2, align="hv")
supp2 <- plot_grid(add_tag(panel_A, "A"), umap_grid, ncol=1,
                   rel_heights=c(0.20, 0.80))

raw_pdf <- tempfile(fileext=".pdf")
exact_pdf <- tempfile(fileext=".pdf")
ggsave(raw_pdf, supp2, width=FIG_W, height=FIG_H, units="mm",
       device=cairo_pdf, bg="white", limitsize=FALSE)
pdf_w_pt <- FIG_W / 25.4 * 72
pdf_h_pt <- FIG_H / 25.4 * 72
gs_status <- system2("gs", c("-q", "-dNOPAUSE", "-dBATCH",
  "-sDEVICE=pdfwrite", sprintf("-dDEVICEWIDTHPOINTS=%.6f", pdf_w_pt),
  sprintf("-dDEVICEHEIGHTPOINTS=%.6f", pdf_h_pt), "-dFIXEDMEDIA",
  "-dPDFFitPage", paste0("-sOutputFile=", exact_pdf), raw_pdf))
stopifnot(gs_status == 0L,
          file.copy(exact_pdf, snakemake@output[["pdf"]], overwrite=TRUE))
unlink(c(raw_pdf, exact_pdf))
ggsave(snakemake@output[["svg"]], supp2, width=FIG_W, height=FIG_H,
       units="mm", device=svglite::svglite, bg="white", limitsize=FALSE)
ggsave(snakemake@output[["png"]], supp2, width=FIG_W, height=FIG_H,
       units="mm", dpi=600, device=ragg::agg_png, bg="white",
       limitsize=FALSE)
cat(sprintf("supp2: %.0f x %.0f mm\n", FIG_W, FIG_H))
options(repr.plot.width=FIG_W/25.4, repr.plot.height=FIG_H/25.4)
supp2
